# Model 1: SentenceTransformer + Regression DNN

**Architecture:** `all-MiniLM-L6-v2` (frozen, 22M params) → 384-dim dense embedding → DNN head (1024-dim, 6 ResidualBlocks, ~13M trainable params)

**Target:** Beat BoW DNN baseline (MAE $46.49)

**Key technique:** Pre-compute embeddings once → train regression head only → fast training

## vast.ai Setup (chỉ chạy lần đầu khi thuê máy)

Sau khi `git clone` repo và `cd` vào đúng thư mục, mở terminal trên vast.ai và chạy:

```bash
pip install uv
uv sync
```

Sau đó khởi động lại Jupyter kernel rồi chạy các cell bên dưới.

In [1]:
from pricer.items import Item
from pricer.sentence_transformer_model import SentTransRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

Pre-computing 800k embeddings with SentenceTransformer. This is the most time-consuming step (~10-15 min on GPU).

In [3]:
runner = SentTransRunner(train, val[:1000])
runner.setup()

Loading SentenceTransformer encoder (frozen)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Pre-computing train embeddings...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Pre-computing val embeddings...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

SentTrans DNN created with 13,017,089 trainable parameters
Using cuda


## 3. Train

Max 15 epochs with early stopping (patience=3). CosineAnnealingLR schedule.

In [4]:
history = runner.train(epochs=15, patience=3)

Epoch 1/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [1/15]
  Train Loss: 0.6032, Val Loss: 0.5284
  Val MAE: $72.79, LR: 0.001000
  ** New best Val MAE: $72.79


Epoch 2/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [2/15]
  Train Loss: 0.4939, Val Loss: 0.4782
  Val MAE: $66.07, LR: 0.000989
  ** New best Val MAE: $66.07


Epoch 3/15:   0%|          | 0/3125 [00:00<?, ?it/s]

...

Epoch [13/15]
  Train Loss: 0.3549, Val Loss: 0.4098
  Val MAE: $56.65, LR: 0.000095
  No improvement (1/3)


Epoch 14/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [14/15]
  Train Loss: 0.3508, Val Loss: 0.4088
  Val MAE: $56.32, LR: 0.000043
  No improvement (2/3)


Epoch 15/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [15/15]
  Train Loss: 0.3487, Val Loss: 0.4088
  Val MAE: $56.24, LR: 0.000011
  ** New best Val MAE: $56.24


## 4. Training History

In [5]:
plot_training_history(history, title="SentenceTransformer + DNN")

## 5. Evaluate on 200 Test Samples

Using `evaluate()` from `pricer/evaluator.py` — same evaluation framework as all other models.

In [6]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$46 $92 $26 $13 $56 $99 $29 $13 $13 $110 $61 $171 $5 $69 $16 $5 $41 $8 $15 $42 $16 $14 $10 $109 $2 $232 $131 $10 $69 $59 $39 $54 $104 $33 $1 $333 $51 $28 $173 $4 $14 $42 $8 $20 $74 $6 $13 $10 $76 $7 $12 $50 $48 $21 $3 $59 $20 $54 $27 $21 $138 $20 $34 $35 $361 $1 $25 $319 $89 $65 $19 $11 $18 $18 $2 $20 $233 $0 $11 $10 $82 $19 $10 $60 $6 $96 $86 $19 $15 $129 $16 $3 $1 $3 $4 $39 $4 $22 $18 $182 $28 $7 $3 $61 $7 $61 $2 $253 $2 $209 $17 $12 $6 $42 $6 $81 $9 $1 $20 $105 $12 $63 $22 $29 $69 $36 $1 $3 $27 $79 $67 $95 $5 $3 $126 $27 $84 $17 $27 $39 $58 $111 $19 $59 $5 $15 $6 $234 $48 $9 $16 $37 $12 $62 $9 $108 $96 $9 $33 $17 $10 $16 $3 $1 $238 $5 $244 $32 $49 $8 $12 $16 $198 $18 $4 $0 $5 $18 $47 $13 $127 $29 $67 $60 $10 $16 $59 $14 $39 $2 $7 $11 $12 $65 $5 $4 $58 $12 $15 $10 

## 6. Save Model Weights

In [7]:
runner.save("sentence_transformer_model.pth")
print("Saved to sentence_transformer_model.pth")

Saved to sentence_transformer_model.pth


# 1. Sanity check — inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred_original:.2f}")
print(f"Error:   ${abs(pred_original - sample.price):.2f}")
print()

# 2. Load roundtrip test — load lại từ .pth và so sánh kết quả
runner.load("sentence_transformer_model.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch! Before=${pred_original:.2f} After=${pred_loaded:.2f}"
print(f"Load roundtrip test PASSED. Diff: ${diff:.4f}")

In [8]:
# Quick sanity check
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $172.61
Error:   $46.39
